# DLGenAI Project — Milestone 1
**Roll No:** 23f3004491

EDA, text processing, TF-IDF baselines and MAP@3 for the Smart MCQ Solver Challenge.
Each cell prints `Q# answer:` for the corresponding form question.

In [1]:
import string

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

BASE = "/kaggle/input/competitions/smart-mcq-solver-challenge"
train = pd.read_csv(f"{BASE}/train.csv")

OPTIONS = ['A', 'B', 'C', 'D', 'E']
print(train.shape, train.columns.tolist())

(2000, 8) ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']


## Q1. Answer frequency distribution
Sum of the most frequent and least frequent option counts.

In [2]:
counts = train['answer'].value_counts()
print(counts.to_string())

print("Q1 answer:", counts.max() + counts.min())

answer
B    490
C    459
A    369
D    358
E    324
Q1 answer: 814


## Q2. Vocabulary size of cleaned prompts

Lowercase, strip `string.punctuation`, split on whitespace, count unique tokens.

In [3]:
def clean_text(text):
    """Lowercase and remove standard punctuation."""
    text = str(text).lower()
    return text.translate(str.maketrans('', '', string.punctuation))

vocabulary = set()
for prompt in train['prompt']:
    vocabulary.update(clean_text(prompt).split())

print("Q2 answer:", len(vocabulary))

Q2 answer: 859


## Q3. Stop-word filtering on Row ID 1

In [4]:
row1 = train[train['id'] == 1].iloc[0]
tokens_row1 = [w for w in clean_text(row1['prompt']).split()
               if w not in ENGLISH_STOP_WORDS]

print(tokens_row1)
print("Q3 answer:", len(tokens_row1))

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
Q3 answer: 13


## Q4. TF-IDF vocabulary size

The vectorizer is fitted on the combined text of each row (prompt + its five options),
i.e. one document per question.

In [5]:
combined_docs = (train['prompt'].astype(str) + " " +
                 train[OPTIONS].astype(str).agg(" ".join, axis=1)).tolist()

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_docs)

print("Q4 answer:", len(vectorizer.get_feature_names_out()))

Q4 answer: 2762


## Q5. Cosine similarity between prompt and option A (Row ID 1)

In [6]:
prompt_vec = vectorizer.transform([str(row1['prompt'])])
option_a_vec = vectorizer.transform([str(row1['A'])])
sim_row1_a = float(cosine_similarity(prompt_vec, option_a_vec)[0][0])

print("Q5 answer:", round(sim_row1_a, 4))

Q5 answer: 0.272


## Q6. Top-1 accuracy of TF-IDF cosine similarity

For every row, rank the five options by cosine similarity to the prompt and check how
often the highest-similarity option is the correct answer.

In [7]:
prompt_matrix = vectorizer.transform(train['prompt'].astype(str))

similarities = np.zeros((len(train), len(OPTIONS)))
for j, option in enumerate(OPTIONS):
    option_matrix = vectorizer.transform(train[option].astype(str))
    similarities[:, j] = cosine_similarity(prompt_matrix, option_matrix).diagonal()

top1 = np.array([OPTIONS[i] for i in similarities.argmax(axis=1)])
accuracy_pct = (top1 == train['answer'].values).mean() * 100

print("Q6 answer:", round(accuracy_pct, 2))

Q6 answer: 13.55


## Q7 & Q8. MAP@3 by hand

Score is `1/(rank+1)` when the correct answer appears in the top-3, else 0.

In [8]:
def average_precision_at_3(true_label, predicted_labels):
    """1.0 at rank 1, 0.5 at rank 2, 1/3 at rank 3, else 0."""
    for rank, pred in enumerate(predicted_labels[:3]):
        if pred == true_label:
            return 1.0 / (rank + 1)
    return 0.0

print("Q7 answer:", average_precision_at_3('C', ['C', 'A', 'B']))
print("Q8 answer:", average_precision_at_3('B', ['D', 'B', 'E']))

Q7 answer: 1.0
Q8 answer: 0.5


## Q9. Majority-class baseline

Predict the three most frequent answers, in frequency order, for every row.

In [9]:
majority_order = counts.index.tolist()[:3]
majority_map3 = np.mean([average_precision_at_3(a, majority_order)
                         for a in train['answer']])

print("static prediction:", " ".join(majority_order))
print("Q9 answer:", round(majority_map3, 4))

static prediction: B C A
Q9 answer: 0.4212


## Q10. TF-IDF pipeline MAP@3

Rank all five options by similarity, take the top three, and average AP@3 over the
training set.

In [10]:
tfidf_predictions = [
    [opt for opt, _ in sorted(zip(OPTIONS, scores), key=lambda pair: -pair[1])]
    for scores in similarities
]
tfidf_map3 = np.mean([average_precision_at_3(a, p)
                      for a, p in zip(train['answer'], tfidf_predictions)])

print("Q10 answer:", round(tfidf_map3, 6))

Q10 answer: 0.296167


## Observations

- Answers are mildly imbalanced (B most frequent, E least), so the majority-class
  baseline already reaches **0.42** MAP@3.
- The TF-IDF pipeline scores **0.296** — *below* the majority baseline. Lexical overlap
  is not merely weak here, it is actively misleading.
- Diagnosis: for a large share of rows every option has **zero** cosine similarity to the
  prompt (no shared non-stop-word), so the ranking is arbitrary. The distractors are also
  near-paraphrases of the correct answer, so word overlap cannot separate them.
- Conclusion: this task needs semantic and knowledge-based methods rather than
  bag-of-words similarity.

In [11]:
zero_similarity_rows = int((similarities.sum(axis=1) == 0).sum())
print("rows where every option has zero similarity:",
      zero_similarity_rows, f"({zero_similarity_rows / len(train) * 100:.1f}%)")

rows where every option has zero similarity: 342 (17.1%)
